# Harness do zero: um agente sem nenhum framework

Este notebook constroi, peca por peca, a infraestrutura minima que transforma um LLM (que so preve texto) em um **agente** que executa acoes reais. Nada de LangChain, LangGraph ou qualquer outro framework — so o loop nu, para deixar claro exatamente o que um framework automatiza depois.

**O que voce vai construir:**
1. Duas ferramentas reais (calculadora segura + consulta de clima)
2. Um *dispatcher* que executa a ferramenta certa e trata erros
3. O **loop do harness**: chama o modelo, interpreta a resposta, executa ferramentas, devolve o resultado, repete — com limite de passos

**Sobre os testes neste notebook**: as secoes 1-3 (ferramentas, dispatcher, loop) sao testadas aqui com um **cliente simulado** (mock), que imita o formato de resposta da API sem precisar de chave nem internet — isso valida a mecanica do harness isoladamente. A secao 5 mostra o mesmo harness plugado na API real da Anthropic, que voce roda localmente com `pip install anthropic` e sua `ANTHROPIC_API_KEY`.

## 1. Ferramentas reais

### Ferramenta 1: calculadora segura

Importante: nunca use `eval()` puro para isso — ele executaria qualquer codigo Python, incluindo coisas maliciosas se o input vier de um LLM (que pode ser manipulado por conteudo externo). Em vez disso, fazemos o parsing da expressao como uma arvore sintatica (AST) e so permitimos operacoes aritmeticas explicitamente.

In [ ]:
import ast
import operator as op

_ALLOWED_OPS = {
    ast.Add: op.add, ast.Sub: op.sub, ast.Mult: op.mul, ast.Div: op.truediv,
    ast.Pow: op.pow, ast.USub: op.neg,
}

def _eval_node(node):
    if isinstance(node, ast.Constant):
        return node.value
    if isinstance(node, ast.BinOp):
        return _ALLOWED_OPS[type(node.op)](_eval_node(node.left), _eval_node(node.right))
    if isinstance(node, ast.UnaryOp):
        return _ALLOWED_OPS[type(node.op)](_eval_node(node.operand))
    raise ValueError(f"Expressao nao suportada: {ast.dump(node)}")

def calculator(expression):
    """Avalia uma expressao aritmetica simples, com seguranca (sem eval())."""
    try:
        tree = ast.parse(expression, mode="eval")
        result = _eval_node(tree.body)
        return {"result": result}
    except Exception as e:
        return {"error": f"Nao foi possivel avaliar \'{expression}\': {e}"}


print(calculator("2 + 2 * 10"))
print(calculator("import os"))  # tentativa de escape -- deve falhar com seguranca, nao executar


### Ferramenta 2: consulta de clima

Em producao, isso seria uma chamada HTTP de verdade (ex: `requests.get(...)` para a API do Open-Meteo, que e gratuita e nao exige chave). Aqui usamos um pequeno banco de dados fixo local, **so para o notebook rodar offline sem depender de rede** — a interface da funcao (recebe uma cidade, devolve um dict) e identica a de uma chamada HTTP real, entao trocar depois e trivial (veja a nota comentada no codigo).

In [ ]:
_FAKE_WEATHER_DB = {
    "sao paulo": {"temp_c": 22, "condicao": "nublado"},
    "rio de janeiro": {"temp_c": 28, "condicao": "ensolarado"},
    "belo horizonte": {"temp_c": 19, "condicao": "chuvoso"},
}

def get_weather(city):
    """Retorna o clima atual de uma cidade.

    Para usar dados reais, troque o corpo desta funcao por algo como:
        import requests
        resp = requests.get("https://api.open-meteo.com/v1/forecast", params={...})
        return resp.json()
    """
    key = city.strip().lower()
    if key not in _FAKE_WEATHER_DB:
        return {"error": f"cidade \'{city}\' nao encontrada na base de dados"}
    return _FAKE_WEATHER_DB[key]


print(get_weather("Sao Paulo"))
print(get_weather("Atlantis"))  # cidade inexistente -- deve devolver erro tratado, nao travar


## 2. Dispatcher: executar a ferramenta certa, com erros tratados

O modelo nunca executa codigo diretamente — ele apenas *sugere* uma chamada (nome da ferramenta + argumentos). O dispatcher e a peca que traduz essa sugestao numa execucao real, e e responsavel por nunca deixar uma excecao Python vazar e quebrar o loop inteiro: todo erro vira uma mensagem estruturada que volta para o modelo, que pode decidir o que fazer a seguir (tentar de novo, pedir mais informacao, desistir).

In [ ]:
TOOL_FUNCTIONS = {
    "calculator": lambda args: calculator(args["expression"]),
    "get_weather": lambda args: get_weather(args["city"]),
}

def execute_tool(tool_name, tool_args):
    if tool_name not in TOOL_FUNCTIONS:
        return {"error": f"ferramenta desconhecida: {tool_name}"}
    try:
        return TOOL_FUNCTIONS[tool_name](tool_args)
    except KeyError as e:
        return {"error": f"argumento obrigatorio ausente: {e}"}
    except Exception as e:
        return {"error": f"erro ao executar {tool_name}: {e}"}


print(execute_tool("calculator", {"expression": "3*7"}))
print(execute_tool("ferramenta_inexistente", {}))       # nome errado -- tratado
print(execute_tool("calculator", {}))                    # argumento faltando -- tratado


## 3. O loop do harness

Este e o coracao do notebook. A cada iteracao:

1. Chama o modelo com o historico de mensagens ate agora
2. Separa a resposta em blocos de texto (raciocinio/resposta final) e blocos de `tool_use` (pedidos de ferramenta)
3. Adiciona a resposta do modelo ao historico
4. Se o modelo nao pediu nenhuma ferramenta (`stop_reason != "tool_use"`), o agente terminou — devolve a resposta final
5. Caso contrario, executa cada ferramenta pedida via `execute_tool`, empacota os resultados no formato que a API espera (`tool_result`) e adiciona ao historico
6. Repete, respeitando um **limite maximo de passos** — sem isso, um modelo que entra em loop (pedindo a mesma ferramenta repetidamente) rodaria para sempre, gastando tokens sem fim

In [ ]:
import json

def run_agent(client, messages, max_steps=5, verbose=True):
    steps = 0
    while steps < max_steps:
        steps += 1
        response = client.create(messages=messages)

        text_blocks = [b for b in response.content if b.type == "text"]
        tool_blocks = [b for b in response.content if b.type == "tool_use"]

        if verbose:
            for b in text_blocks:
                print(f"[passo {steps}] modelo diz: {b.text}")

        messages.append({"role": "assistant", "content": response.content})

        if response.stop_reason != "tool_use":
            final_text = " ".join(b.text for b in text_blocks)
            return {"status": "concluido", "resposta": final_text, "passos": steps}

        tool_results = []
        for block in tool_blocks:
            if verbose:
                print(f"[passo {steps}] chamando ferramenta: {block.name}({block.input})")
            result = execute_tool(block.name, block.input)
            tool_results.append({
                "type": "tool_result",
                "tool_use_id": block.id,
                "content": json.dumps(result, ensure_ascii=False),
            })

        messages.append({"role": "user", "content": tool_results})

    return {"status": "limite_de_passos_atingido", "passos": steps}


## 4. Testando o loop com um cliente simulado (mock)

Antes de gastar chamadas de API de verdade, vale testar a **mecanica do loop** com respostas simuladas — isso separa bugs no seu harness de comportamento inesperado do modelo. As classes abaixo imitam exatamente o formato de resposta da API da Anthropic (`content`, `stop_reason`, blocos `text`/`tool_use`), mas com respostas roteirizadas por nos.

In [ ]:
class FakeToolUseBlock:
    def __init__(self, id, name, input_):
        self.type = "tool_use"
        self.id = id
        self.name = name
        self.input = input_

class FakeTextBlock:
    def __init__(self, text):
        self.type = "text"
        self.text = text

class FakeResponse:
    def __init__(self, content, stop_reason):
        self.content = content
        self.stop_reason = stop_reason

class FakeAnthropicClient:
    """Devolve uma sequencia pre-definida de respostas, uma a cada chamada -- util para testar o loop sem API real."""
    def __init__(self, scripted_responses):
        self.scripted_responses = scripted_responses
        self.call_count = 0

    def create(self, **kwargs):
        response = self.scripted_responses[self.call_count]
        self.call_count += 1
        return response


**Cenario 1 — caminho feliz**: o modelo pede a calculadora, recebe o resultado, e responde.

In [ ]:
scripted = [
    FakeResponse([FakeToolUseBlock("t1", "calculator", {"expression": "12*7"})], "tool_use"),
    FakeResponse([FakeTextBlock("O resultado de 12*7 e 84.")], "end_turn"),
]
client = FakeAnthropicClient(scripted)
resultado = run_agent(client, [{"role": "user", "content": "quanto e 12 vezes 7?"}])
print("\nResultado final:", resultado)


**Cenario 2 — ferramenta inexistente**: o modelo "alucina" uma ferramenta que nao existe. O harness nao deve travar — o erro deve voltar ao modelo, que decide como reagir.

In [ ]:
scripted2 = [
    FakeResponse([FakeToolUseBlock("t1", "ferramenta_fantasma", {})], "tool_use"),
    FakeResponse([FakeTextBlock("Desculpe, nao consegui completar isso.")], "end_turn"),
]
client2 = FakeAnthropicClient(scripted2)
resultado2 = run_agent(client2, [{"role": "user", "content": "faca algo impossivel"}])
print("\nResultado final:", resultado2)


**Cenario 3 — loop infinito**: o modelo fica pedindo a mesma ferramenta indefinidamente (simulando um modelo travado ou um prompt mal projetado). O `max_steps` deve interromper a execucao em vez de rodar para sempre.

In [ ]:
scripted3 = [
    FakeResponse([FakeToolUseBlock(f"t{i}", "calculator", {"expression": "1+1"})], "tool_use")
    for i in range(10)
]
client3 = FakeAnthropicClient(scripted3)
resultado3 = run_agent(client3, [{"role": "user", "content": "loop infinito"}], max_steps=3, verbose=False)
print("Resultado final:", resultado3)
assert resultado3["status"] == "limite_de_passos_atingido"
print("Limite de passos respeitado corretamente.")


## 5. O mesmo harness com a API real

Agora trocamos o `FakeAnthropicClient` pelo cliente real da Anthropic. A logica do `run_agent` **nao muda nada** — essa e a prova de que o mock testou a mecanica certa: o harness e agnostico a se o modelo e simulado ou real, desde que a interface (`'.create(messages=...)'` devolvendo `.content` e `.stop_reason`) seja compatível.

Precisamos definir os *schemas* das ferramentas (formato JSON Schema) para o modelo saber que ferramentas existem e como chama-las — isso e o unico pedaco novo.

**Requer**: `pip install anthropic` e `ANTHROPIC_API_KEY` configurada.

In [ ]:
import os
from anthropic import Anthropic

TOOLS_SCHEMA = [
    {
        "name": "calculator",
        "description": "Avalia uma expressao aritmetica simples (soma, subtracao, multiplicacao, divisao, potencia).",
        "input_schema": {
            "type": "object",
            "properties": {
                "expression": {"type": "string", "description": "Expressao matematica, ex: \'12 * 7\'"}
            },
            "required": ["expression"],
        },
    },
    {
        "name": "get_weather",
        "description": "Retorna a temperatura e condicao do tempo atual de uma cidade.",
        "input_schema": {
            "type": "object",
            "properties": {
                "city": {"type": "string", "description": "Nome da cidade, ex: \'Sao Paulo\'"}
            },
            "required": ["city"],
        },
    },
]

raw_client = Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY"))

class RealAnthropicClient:
    """Encapsula o cliente real para ter a mesma interface .create(messages=...) usada pelo mock."""
    def create(self, messages):
        return raw_client.messages.create(
            model="claude-sonnet-4-6",
            max_tokens=1024,
            tools=TOOLS_SCHEMA,
            messages=messages,
        )


real_client = RealAnthropicClient()
resultado_real = run_agent(
    real_client,
    [{"role": "user", "content": "Qual e o clima em Sao Paulo, e quanto e 15% desse valor de temperatura em graus?"}],
    max_steps=6,
)
print("\nResultado final:", resultado_real)


## Notas finais e proximos passos

- **O que o harness manual deixou explicito**: parsing de `tool_use`, dispatch com tratamento de erro, gerenciamento de historico (`messages`), e limite de passos — tudo isso e o que frameworks como LangGraph fazem por baixo dos panos. Agora que voce escreveu isso a mao, o proximo projeto (reimplementar o mesmo agente em LangGraph) vai deixar claro exatamente o que o framework esta automatizando
- **Melhorias possiveis neste harness**:
  - **Timeout por passo**: alem do limite de numero de passos, limitar o tempo total de execucao
  - **Orcamento de custo**: somar os tokens gastos a cada chamada e parar se ultrapassar um limite
  - **Logging estruturado**: salvar cada passo (ferramenta chamada, argumentos, resultado, tempo) para depois auditar ou avaliar a trajetoria do agente (conecta com a Semana 6 desta fase)
  - **Retry com backoff**: se uma ferramenta falhar por um erro transitorio (ex: timeout de rede), tentar de novo antes de desistir
- **Seguranca**: note que a calculadora usa parsing por AST, nunca `eval()` puro — o mesmo cuidado vale para qualquer ferramenta que execute codigo ou comandos de sistema a partir de um input que passou pelo modelo, ja que esse input pode ter sido influenciado por conteudo externo nao confiavel (prompt injection, tema da Fase 7)
